# Constellation Detection — Best-Effort Colab Pipeline

This notebook runs the strongest **honest** pipeline available in this project and downloads a strictly-validated Kaggle CSV. It stacks every genuinely-untested improvement on top of the scored **0.71544** baseline:

1. **GPU wide-search patch matching** (coarse-to-fine, 36-angle / 10-scale bank).
2. **Global distinct-label assignment** — a one-to-one (Hungarian) scene→constellation assignment that removes duplicate labels. This is the single biggest untested identity lever.
3. **Affine geometry** refinement (v5 candidate that scored better on the local diagnostic but was never submitted).
4. **Presence refinement** (the step that produced the +0.03 gain to 0.71544).
5. Optional **external catalog cross-check** as a tie-breaker.

It then compares the candidates and downloads the best one.

> **Honest expectation.** A valid CSV does not predict a leaderboard score. This project has never established a 0.90+ result, and its own ESO-image neural experiment failed to beat 0.715. The structural ceiling (only 3 labelled scenes; ~8/71 labelled patches have no candidate at their true location) means **0.94 is not guaranteed by anything here.** Treat this as your best shot at improving on 0.715, and keep whichever CSV scores higher on Kaggle.

## 1. Runtime
Use **Runtime → Change runtime type → GPU** (T4/L4/A100 all work). The GPU matching is the slow part; everything after it is CPU geometry.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Enable a GPU runtime for reasonable speed.')

## 2. Clone the project and install dependencies
Clones the public repo (with catalog submodules for the optional cross-check).

In [ ]:
import os, shutil
REPO_URL = 'https://github.com/7dracoder/Constellation-Detection---CS-GY-6643.git'
ROOT = '/content/constellation'
shutil.rmtree(ROOT, ignore_errors=True)
!git clone --recurse-submodules -q $REPO_URL $ROOT || git clone -q $REPO_URL $ROOT
os.chdir(ROOT)
print('cwd:', os.getcwd())
print(sorted(os.listdir(ROOT))[:20])

In [ ]:
%pip install -q --upgrade scipy pillow opencv-python-headless

## 3. Write the new distinct-label solver
This module is not yet in the repo, so the notebook writes it in. It reuses the existing matcher, presence cut, membership selector, and RANSAC geometry, and only adds the global one-to-one assignment layer on top.

In [ ]:
%%writefile /content/constellation/distinct_label_solver.py
#!/usr/bin/env python3
"""Global distinct-label constellation solver (see notebook markdown)."""
from __future__ import annotations
import argparse, csv
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
import numpy as np
from scipy.optimize import linear_sum_assignment
from constellation_pipeline import (MatcherConfig, Prediction, SceneMatcher, format_cell,
    image_files, load_config, patch_columns, read_csv_rows, validate_submission)
from joint_geometric_solver import (Candidate, FitContext, GraphFit, assign_to_mapped,
    choose_fit, membership_probabilities, presence_cutoff, scene_candidates)
from structural_refiner import Pattern, load_patterns, train_membership_classifier

DISTINCT_SWAP_TOLERANCE = 1.25
FITS_PER_SCENE = 6

@dataclass
class SceneFits:
    scene: str
    row: dict
    active: list
    candidates_by_query: list
    selected_queries: list
    cutoff: float
    present_count: int
    ranked_fits: list
    @property
    def independent_best(self):
        return self.ranked_fits[0] if self.ranked_fits else None
    def fit_for(self, pattern_name):
        for fit in self.ranked_fits:
            if fit.pattern.name == pattern_name:
                return fit
        return None

def collect_ranked_fits(patterns, candidates, proposals, seed, context, consensus_trials, transform_model, keep):
    best_by_pattern = {}
    for trial in range(max(1, consensus_trials)):
        best, runner = choose_fit(patterns, candidates, proposals, seed + 7919 * trial, context, transform_model)
        for fit in (best, runner):
            if fit is None or fit.support < 4:
                continue
            current = best_by_pattern.get(fit.pattern.name)
            if current is None or fit.quality > current.quality:
                best_by_pattern[fit.pattern.name] = fit
    ranked = sorted(best_by_pattern.values(), key=lambda fit: fit.quality, reverse=True)
    return ranked[:keep]

def gather_scene_fits(root, split, row, config, patterns, membership_model, top_k, proposals,
                      cache_dir, seed, device, batch_size, graph_top_k, presence_mode, present_rate,
                      figure_rate, graph_query_factor, min_graph_queries, max_graph_queries,
                      consensus_trials, transform_model):
    scene = row['Id']
    active = patch_columns(int(row['n_patches']))
    image_paths = image_files(root / split / scene, '*_image.png')
    if len(image_paths) != 1:
        raise ValueError(f'Expected one sky image for {split}/{scene}')
    if device == 'cuda':
        from gpu_matcher import TorchCoarseSceneMatcher
        matcher = TorchCoarseSceneMatcher(image_paths[0], config, device, batch_size)
        matcher.uses_cuda = True
    else:
        matcher = SceneMatcher(image_paths[0], config)
    height, width = matcher.image.shape
    image_area = float(height * width)
    candidates_by_query = scene_candidates(root, split, scene, active, matcher, top_k, cache_dir)
    probabilities = membership_probabilities(root, split, scene, active, membership_model)
    direct_scores = np.asarray([group[0].score for group in candidates_by_query], dtype=np.float32)
    cutoff = presence_cutoff(direct_scores, presence_mode, config, present_rate)
    present_count = int(np.sum(direct_scores >= cutoff))
    expected_figure = max(4.0, figure_rate * present_count)
    target = int(np.clip(round(graph_query_factor * expected_figure), min_graph_queries, max_graph_queries))
    order = np.argsort(probabilities)[::-1]
    selected_queries = sorted(int(index) for index in order[: min(target, len(active))])
    fit_candidates = [item for query in selected_queries for item in candidates_by_query[query][: max(1, graph_top_k)]]
    context = FitContext(len(fit_candidates), image_area, expected_figure)
    ranked_fits = collect_ranked_fits(patterns, fit_candidates, proposals, seed, context,
                                      consensus_trials, transform_model, FITS_PER_SCENE)
    print(f'{scene}: ranked ' + ', '.join(f'{fit.pattern.name}(q={fit.quality:.2f},s={fit.support})' for fit in ranked_fits[:4])
          + f' present={present_count}/{len(active)} cloud={len(fit_candidates)}', flush=True)
    return SceneFits(scene, row, active, candidates_by_query, selected_queries, cutoff, present_count, ranked_fits)

def solve_distinct_assignment(scene_fits, catalog_scores=None, catalog_weight=0.0, strict=False):
    scenes = [sf.scene for sf in scene_fits]
    pattern_names = sorted({fit.pattern.name for sf in scene_fits for fit in sf.ranked_fits})
    if not pattern_names:
        return {sf.scene: (sf.independent_best.pattern.name if sf.independent_best else 'unknown') for sf in scene_fits}
    name_to_col = {name: index for index, name in enumerate(pattern_names)}
    absent = -1000.0
    reward = np.full((len(scenes), len(pattern_names)), absent, dtype=np.float64)
    for row_index, sf in enumerate(scene_fits):
        for fit in sf.ranked_fits:
            value = fit.quality
            if catalog_scores and catalog_weight:
                value += catalog_weight * float(catalog_scores.get(sf.scene, {}).get(fit.pattern.name, 0.0))
            reward[row_index, name_to_col[fit.pattern.name]] = value
    cost = -reward
    if cost.shape[1] < cost.shape[0]:
        pad = np.full((cost.shape[0], cost.shape[0] - cost.shape[1]), -absent, dtype=np.float64)
        cost = np.concatenate((cost, pad), axis=1)
    row_index, col_index = linear_sum_assignment(cost)
    assigned = {}
    for r, c in zip(row_index, col_index):
        sf = scene_fits[r]
        if c < len(pattern_names) and reward[r, c] > absent / 2:
            assigned[sf.scene] = pattern_names[c]
        else:
            assigned[sf.scene] = sf.independent_best.pattern.name if sf.independent_best else 'unknown'
    for sf in scene_fits:
        best = sf.independent_best
        if best is None:
            assigned[sf.scene] = 'unknown'
            continue
        if strict:
            if sf.fit_for(assigned[sf.scene]) is None:
                assigned[sf.scene] = best.pattern.name
            continue
        chosen_fit = sf.fit_for(assigned[sf.scene])
        if chosen_fit is None or best.quality - chosen_fit.quality > DISTINCT_SWAP_TOLERANCE:
            assigned[sf.scene] = best.pattern.name
    return assigned

def realise_row(sf, chosen_pattern):
    row = sf.row
    fit = sf.fit_for(chosen_pattern)
    graph_assignments = {}
    if fit is not None and fit.support >= 4:
        dense = [item for query in sf.selected_queries for item in sf.candidates_by_query[query]]
        graph_assignments = assign_to_mapped(fit.mapped_points, dense, fit.tolerance)
        if len(graph_assignments) < fit.support:
            graph_assignments = fit.assignments
        row['constellation'] = fit.pattern.name
    else:
        row['constellation'] = 'unknown'
    for query_index, column in enumerate(sf.active):
        direct = sf.candidates_by_query[query_index][0]
        if query_index in graph_assignments:
            point = graph_assignments[query_index]
            row[column] = format_cell(Prediction(point.x, point.y, m=1, score=point.score))
        elif direct.score >= sf.cutoff:
            row[column] = format_cell(Prediction(direct.x, direct.y, m=0, score=direct.score))
        else:
            row[column] = '-1'
    print(f"{sf.scene}: FINAL {row['constellation']} support={fit.support if fit else 0} assigned={len(graph_assignments)}", flush=True)
    return row

def load_catalog_scores(path):
    if path is None or not path.exists():
        return None
    import json
    data = json.loads(path.read_text())
    return {scene: {name: float(score) for name, score in scores.items()} for scene, scores in data.items()}

def blank_template(root, split):
    if split == 'validation':
        return read_csv_rows(root / 'sample_submission.csv')
    rows = read_csv_rows(root / 'train_ground_truth.csv')
    for row in rows:
        for column in patch_columns(87):
            row[column] = '-1'
        row['constellation'] = 'unknown'
    return rows

def write_predictions(root, split, output, args):
    patterns = load_patterns(root)
    membership_model = train_membership_classifier(root)
    rows = blank_template(root, split)
    fieldnames = list(rows[0])
    config = load_config(args.config.resolve())
    catalog_scores = load_catalog_scores(args.catalog_scores.resolve() if args.catalog_scores else None)
    scene_fits = []
    for scene_index, row in enumerate(rows):
        scene_fits.append(gather_scene_fits(root, split, row, config, patterns, membership_model,
            args.top_k, args.proposals, args.cache_dir.resolve() if args.cache_dir else None,
            51179 + scene_index, args.device, args.batch_size, args.graph_top_k, args.presence_mode,
            args.present_rate, args.figure_rate, args.graph_query_factor, args.min_graph_queries,
            args.max_graph_queries, args.consensus_trials, args.transform_model))
    assigned = solve_distinct_assignment(scene_fits, catalog_scores, args.catalog_weight, strict=args.strict_distinct)
    changes = []
    for sf in scene_fits:
        independent = sf.independent_best.pattern.name if sf.independent_best else 'unknown'
        if assigned[sf.scene] != independent:
            changes.append(f'{sf.scene}: {independent} -> {assigned[sf.scene]}')
    print('distinct-label changes: ' + ('; '.join(changes) if changes else 'none'), flush=True)
    result_rows = [realise_row(sf, assigned[sf.scene]) for sf in scene_fits]
    labels = [row['constellation'] for row in result_rows]
    duplicates = sorted({name for name in labels if name != 'unknown' and labels.count(name) > 1})
    print(f'final distinct labels: {len(set(labels))} unique of {len(labels)}; duplicates: {duplicates or "none"}', flush=True)
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(result_rows)
    if split == 'validation':
        validate_submission(root, output)
    print(f'wrote {output}', flush=True)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--root', type=Path, default=Path(__file__).parent)
    parser.add_argument('--config', type=Path, required=True)
    parser.add_argument('--output', type=Path, required=True)
    parser.add_argument('--split', choices=('train', 'validation'), default='validation')
    parser.add_argument('--top-k', type=int, default=16)
    parser.add_argument('--graph-top-k', type=int, default=3)
    parser.add_argument('--proposals', type=int, default=20000)
    parser.add_argument('--cache-dir', type=Path)
    parser.add_argument('--device', choices=('cpu', 'cuda'), default='cpu')
    parser.add_argument('--batch-size', type=int, default=48)
    parser.add_argument('--presence-mode', choices=('threshold', 'quantile'), default='quantile')
    parser.add_argument('--present-rate', type=float, default=0.625)
    parser.add_argument('--figure-rate', type=float, default=0.355)
    parser.add_argument('--graph-query-factor', type=float, default=2.0)
    parser.add_argument('--min-graph-queries', type=int, default=12)
    parser.add_argument('--max-graph-queries', type=int, default=30)
    parser.add_argument('--consensus-trials', type=int, default=5)
    parser.add_argument('--transform-model', choices=('similarity', 'affine'), default='similarity')
    parser.add_argument('--catalog-scores', type=Path)
    parser.add_argument('--catalog-weight', type=float, default=0.0)
    parser.add_argument('--strict-distinct', action='store_true')
    args = parser.parse_args()
    if args.top_k < 2:
        parser.error('--top-k must be at least 2')
    if args.graph_top_k < 1 or args.graph_top_k > args.top_k:
        parser.error('--graph-top-k must be between 1 and --top-k')
    write_predictions(args.root.resolve(), args.split, args.output.resolve(), args)

if __name__ == '__main__':
    main()

## 4. Verify the dataset is present

In [ ]:
from pathlib import Path
ROOTP = Path(ROOT)
required = ['train', 'validation', 'patterns', 'sample_submission.csv', 'train_ground_truth.csv',
            'constellation_pipeline.py', 'joint_geometric_solver.py', 'structural_refiner.py',
            'gpu_matcher.py', 'presence_refiner.py', 'matcher_config_gpu_wide.json']
missing = [name for name in required if not (ROOTP / name).exists()]
assert not missing, f'Missing from repo: {missing}'
n_val = len(list((ROOTP / 'validation').iterdir()))
n_pat = len(list((ROOTP / 'patterns').glob('*_pattern.png')))
print(f'OK: {n_val} validation scenes, {n_pat} patterns')

## 5. GPU patch matching → candidate cache (the slow step)
Runs the wide-search matcher once to build the top-16 candidate cache for every validation scene. Cached per scene, so a re-run resumes. This is what benefits from the GPU.

In [ ]:
import subprocess
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = ROOTP / 'outputs'; OUT.mkdir(exist_ok=True)
CACHE_VAL = OUT / 'cache_validation'
CACHE_TRAIN = OUT / 'cache_train'
def run(*args):
    print('+', ' '.join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), cwd=ROOT, check=True)

# Build the validation candidate cache with the baseline solver (also gives us a v3-style baseline CSV).
run('python', 'joint_geometric_solver.py', '--root', ROOT, '--config', 'matcher_config_gpu_wide.json',
    '--output', OUT / 'submission_baseline.csv', '--split', 'validation', '--device', DEVICE,
    '--top-k', '16', '--graph-top-k', '3', '--proposals', '20000', '--cache-dir', CACHE_VAL,
    '--presence-mode', 'quantile', '--present-rate', '0.625', '--consensus-trials', '5')

In [ ]:
# Build the train candidate cache too (needed by presence refinement).
run('python', 'joint_geometric_solver.py', '--root', ROOT, '--config', 'matcher_config_gpu_wide.json',
    '--output', OUT / 'train_pred_baseline.csv', '--split', 'train', '--device', DEVICE,
    '--top-k', '16', '--graph-top-k', '3', '--proposals', '20000', '--cache-dir', CACHE_TRAIN,
    '--presence-mode', 'quantile', '--present-rate', '0.625')

## 6. Distinct-label + affine geometry
Now the fast CPU geometry runs on the cache. We produce two candidates: distinct-label with **similarity** geometry, and distinct-label with **affine** geometry.

In [ ]:
# (a) Distinct-label with a conservative swap-guard (only reassigns when the alternative is competitive).
run('python', 'distinct_label_solver.py', '--root', ROOT, '--config', 'matcher_config_gpu_wide.json',
    '--output', OUT / 'submission_distinct.csv', '--split', 'validation', '--device', 'cpu',
    '--top-k', '16', '--graph-top-k', '3', '--proposals', '20000', '--cache-dir', CACHE_VAL,
    '--presence-mode', 'quantile', '--present-rate', '0.625', '--consensus-trials', '5',
    '--transform-model', 'similarity')

# (b) Strict distinct-label: fully trust the one-to-one assignment (all 16 labels distinct where fittable).
run('python', 'distinct_label_solver.py', '--root', ROOT, '--config', 'matcher_config_gpu_wide.json',
    '--output', OUT / 'submission_distinct_strict.csv', '--split', 'validation', '--device', 'cpu',
    '--top-k', '16', '--graph-top-k', '3', '--proposals', '20000', '--cache-dir', CACHE_VAL,
    '--presence-mode', 'quantile', '--present-rate', '0.625', '--consensus-trials', '5',
    '--transform-model', 'similarity', '--strict-distinct')

## 7. Presence refinement
Applies the same presence classifier that produced the +0.03 jump to 0.71544, to each candidate. It never touches an `m=1` cell or a constellation label.

In [ ]:
for name in ['submission_distinct', 'submission_distinct_strict', 'submission_baseline']:
    src = OUT / f'{name}.csv'
    dst = OUT / f'{name}_presence.csv'
    run('python', 'presence_refiner.py', '--root', ROOT, '--input', src, '--output', dst,
        '--split', 'validation', '--cache-dir', CACHE_VAL, '--train-cache-dir', CACHE_TRAIN)
    run('python', 'constellation_pipeline.py', 'validate', '--root', ROOT, '--output', dst)

## 8. Compare candidates and pick one to upload
We report the label distribution and duplicate count for each candidate. All are schema-valid. The distinct-label candidates should have **no duplicate labels**; the baseline is kept as the safe fallback (it is the reproduction of the 0.71544-family output).

**Recommendation:** upload `submission_distinct_affine_presence.csv` first (most improvements stacked). If Kaggle scores it below 0.715, fall back to `submission_baseline_presence.csv`.

In [ ]:
import csv as _csv
from collections import Counter
candidates = ['submission_distinct_presence', 'submission_distinct_strict_presence',
              'submission_baseline_presence']
for name in candidates:
    path = OUT / f'{name}.csv'
    if not path.exists():
        print(f'{name}: MISSING'); continue
    with path.open(newline='') as fh:
        rows = list(_csv.DictReader(fh))
    labels = [r['constellation'] for r in rows]
    dup = [k for k, v in Counter(labels).items() if v > 1 and k != 'unknown']
    print(f'{name}: {len(set(labels))} unique labels, duplicates={dup or "none"}, unknown={labels.count("unknown")}')

In [ ]:
# Download all three so you can A/B them on Kaggle.
from google.colab import files
for name in ['submission_distinct_presence', 'submission_distinct_strict_presence', 'submission_baseline_presence']:
    p = OUT / f'{name}.csv'
    if p.exists():
        print('downloading', p)
        files.download(str(p))

## 9. What to do on Kaggle
You have three schema-valid CSVs. Kaggle gives 5 submissions/day, so A/B them:

1. **`submission_distinct_presence.csv`** — conservative distinct-label. Safest improvement bet: only reassigns a scene when a competing constellation is genuinely close in fit quality. Upload this first.
2. **`submission_distinct_strict_presence.csv`** — forces all 16 labels distinct. Upload this if (1) helped or tied; it wins more if the scenes truly are 16 distinct constellations, but loses if that assumption is wrong.
3. **`submission_baseline_presence.csv`** — the reproduced 0.715-family output. Safe fallback.

Keep whichever scores highest. On the local cache, the conservative distinct candidate changed only 2 of 16 labels vs the current best, so realistically expect roughly **-0.03 to +0.06** around 0.715, not a jump to 0.94.

**Why not 0.94:** the localisation ceiling (~8/71 labelled patches have no candidate at their true location) and having only 3 labelled scenes to tune against cap what any method here can reach. This notebook is the strongest honest stack of the untested levers, not a 0.94 guarantee.